In [1]:
#!/usr/bin/env python3
"""
把dynamic_model.m中重力项的表达式线性化，得到参数向量theta和回归矩阵Y(q)，满足tau_g = Y(q) * theta。
"""

from sympy import symbols, cos, Matrix, zeros, pprint

# joint variables
q2, q3 = symbols("q2 q3")

# parameters to identify
b1, b2 = symbols("b1 b2")
theta = Matrix([b1, b2])

# gravity vector from dynamic_model.m
g = Matrix([
    0,
    b1 * cos(q2) + b2 * cos(q2 + q3),
    b2 * cos(q2 + q3),
])

# build regressor Y such that g = Y * theta
# Y_ij = diff(g_i, theta_j)
Y = zeros(3, 2)
for i in range(3):
    for j in range(2):
        Y[i, j] = g[i].diff(theta[j])

print("=== parameter vector theta ===")
pprint(theta.T)

print("\n=== gravity regressor  Y(q) ===")
pprint(Y)

# verification:  Y * theta  should equal  g
print("\n=== verification: Y * theta ===")
pprint(Y * theta)

print("\n=== original g ===")
pprint(g)

assert (Y * theta - g).equals(zeros(3, 1)), "Linearization mismatch!"
print("\nVerification passed: Y * theta == g")


=== parameter vector theta ===
[b₁  b₂]

=== gravity regressor  Y(q) ===
⎡   0          0      ⎤
⎢                     ⎥
⎢cos(q₂)  cos(q₂ + q₃)⎥
⎢                     ⎥
⎣   0     cos(q₂ + q₃)⎦

=== verification: Y * theta ===
⎡             0              ⎤
⎢                            ⎥
⎢b₁⋅cos(q₂) + b₂⋅cos(q₂ + q₃)⎥
⎢                            ⎥
⎣      b₂⋅cos(q₂ + q₃)       ⎦

=== original g ===
⎡             0              ⎤
⎢                            ⎥
⎢b₁⋅cos(q₂) + b₂⋅cos(q₂ + q₃)⎥
⎢                            ⎥
⎣      b₂⋅cos(q₂ + q₃)       ⎦

Verification passed: Y * theta == g


In [2]:
import numpy as np
from sklearn.linear_model import LinearRegression

def gravity_regressor(q2_val, q3_val):
  """
  Compute the numerical gravity regressor Y(q) for given joint angles.
  
  Parameters
  ----------
  q2_val : float or array-like
    Joint angle q2 (rad).
  q3_val : float or array-like
    Joint angle q3 (rad).
  
  Returns
  -------
  Y_num : ndarray, shape (3, 2)
    Numerical gravity regressor matrix.
  """
  c2 = np.cos(q2_val)
  c23 = np.cos(q2_val + q3_val)
  Y_num = np.array([
    [0.0,  0.0],
    [c2,   c23],
    [0.0,  c23],
  ])
  return Y_num


def build_regression_data(q2_arr, q3_arr, tau_g_arr):
  """
  Stack multiple measurements into regression form:  tau_g = Y_stack * theta.
  
  Parameters
  ----------
  q2_arr : array-like, shape (N,)
    Measured q2 values.
  q3_arr : array-like, shape (N,)
    Measured q3 values.
  tau_g_arr : array-like, shape (N, 3)
    Measured gravity torque vectors (each row is [tau1, tau2, tau3]).
  
  Returns
  -------
  Y_stack : ndarray, shape (3*N, 2)
  tau_stack : ndarray, shape (3*N,)
  """
  N = len(q2_arr)
  Y_stack = np.vstack([gravity_regressor(q2_arr[i], q3_arr[i]) for i in range(N)])
  tau_stack = np.array(tau_g_arr).reshape(-1)
  return Y_stack, tau_stack


def identify_gravity_params(q2_arr, q3_arr, tau_g_arr):
  """
  Use least-squares regression (sklearn) to identify gravity parameters [b1, b2].
  
  Parameters
  ----------
  q2_arr : array-like, shape (N,)
  q3_arr : array-like, shape (N,)
  tau_g_arr : array-like, shape (N, 3)
  
  Returns
  -------
  theta_hat : ndarray, shape (2,)
    Estimated parameters [b1, b2].
  score : float
    R² score of the fit.
  """
  Y_stack, tau_stack = build_regression_data(q2_arr, q3_arr, tau_g_arr)
  
  reg = LinearRegression(fit_intercept=False)
  reg.fit(Y_stack, tau_stack)
  
  theta_hat = reg.coef_
  score = reg.score(Y_stack, tau_stack)
  
  print(f"Identified parameters:  b1 = {theta_hat[0]:.6f},  b2 = {theta_hat[1]:.6f}")
  print(f"R² score: {score:.6f}")
  return theta_hat, score


# ===== Demo with synthetic data =====
np.random.seed(42)

# Ground-truth parameters
b1_true, b2_true = 3.5, 1.2
theta_true = np.array([b1_true, b2_true])

# Generate random joint angle measurements
N_samples = 100
q2_data = np.random.uniform(-np.pi, np.pi, N_samples)
q3_data = np.random.uniform(-np.pi, np.pi, N_samples)

# Compute noise-free gravity torques and add Gaussian noise
tau_g_data = np.array([gravity_regressor(q2_data[i], q3_data[i]) @ theta_true
             for i in range(N_samples)])
noise_std = 0.05
tau_g_data += np.random.randn(*tau_g_data.shape) * noise_std

# Identify parameters
theta_hat, r2 = identify_gravity_params(q2_data, q3_data, tau_g_data)

print(f"\nGround truth:  b1 = {b1_true},  b2 = {b2_true}")
print(f"Estimation error:  Δb1 = {abs(theta_hat[0] - b1_true):.6f},  Δb2 = {abs(theta_hat[1] - b2_true):.6f}")

Identified parameters:  b1 = 3.501416,  b2 = 1.198976
R² score: 0.999001

Ground truth:  b1 = 3.5,  b2 = 1.2
Estimation error:  Δb1 = 0.001416,  Δb2 = 0.001024


In [3]:
import pandas as pd

# 读取采集的数据
df = pd.read_csv("data_20260503_011832.csv")
print(f"共 {len(df)} 组数据\n")
print(df[["q2", "q3", "tau1", "tau2", "tau3"]].describe())

# 提取关节角度和力矩
q2_arr = df["q2"].values
q3_arr = df["q3"].values
tau_g_arr = df[["tau1", "tau2", "tau3"]].values

# 辨识 b1, b2
theta_hat, r2 = identify_gravity_params(q2_arr, q3_arr, tau_g_arr)

# 与理论值对比 (b1 = (m2*r2 + m3*l2)*g0,  b2 = m3*r3*g0)
m2, m3 = 1.512, 1.7
l2 = 0.29781
r2_len, r3_len = l2 / 2, 0.3
g0 = 9.8
b1_nominal = (m2 * r2_len + m3 * l2) * g0
b2_nominal = m3 * r3_len * g0
print(f"\n理论标称值:  b1 = {b1_nominal:.4f},  b2 = {b2_nominal:.4f}")
print(f"辨识结果:    b1 = {theta_hat[0]:.4f},  b2 = {theta_hat[1]:.4f}")

共 22 组数据

              q2         q3  tau1       tau2       tau3
count  22.000000  22.000000  22.0  22.000000  22.000000
mean    2.080161  -1.481768   0.0  -0.320280   1.085359
std     0.236406   0.746454   0.0   1.337460   0.800522
min     1.522832  -2.606982   0.0  -2.938458  -0.559218
25%     1.936688  -2.077971   0.0  -1.061541   0.528694
50%     2.180047  -1.592829   0.0   0.061540   1.465202
75%     2.279870  -0.987428   0.0   0.353847   1.656899
max     2.365117  -0.185375   0.0   2.292309   1.789988
Identified parameters:  b1 = 3.119436,  b2 = 1.827563
R² score: 0.974303

理论标称值:  b1 = 7.1679,  b2 = 4.9980
辨识结果:    b1 = 3.1194,  b2 = 1.8276
